In [2]:
import pandas as pd
from pathlib import Path

pasta = Path("../data/raw")

ficheiros = {
    "customer": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

dados = {name: pd.read_csv(pasta / f) for name, f in ficheiros.items()}

for name, df in dados.items():
    print(f"{name}: {df.shape}")




customer: (99441, 5)
geolocation: (1000163, 5)
orders: (99441, 8)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [3]:
orders = dados["orders"]

lista = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]

for col in lista:
    orders[col] = pd.to_datetime(orders[col])

print(orders.dtypes) 
print("\n")

order_items = dados["order_items"]
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

print(order_items.dtypes)
print("\n")

order_reviews = dados["order_reviews"]
lista_review = ["review_creation_date", "review_answer_timestamp"]

for col in lista_review:
    order_reviews[col] = pd.to_datetime(order_reviews[col])


print(order_reviews.dtypes)





order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object


review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [4]:
for name, df in dados.items():                  #Orders, order_reviews and products have null values
    print(f"=== Valores Nulos em {name} ===")
    print(df.isna().sum())



=== Valores Nulos em customer ===
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
=== Valores Nulos em geolocation ===
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
=== Valores Nulos em orders ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
=== Valores Nulos em order_items ===
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
=== Valores Nu

In [5]:
no_delivered_date = orders[orders["order_delivered_customer_date"].isna()]
count = no_delivered_date["order_status"].value_counts()
print(len(no_delivered_date))

pd.DataFrame({"N": count , "%": count/len(no_delivered_date) *100})


2965


,N,%
order_status,,
shipped,1107,37.335582
canceled,619,20.876897
unavailable,609,20.539629
invoiced,314,10.590219
processing,301,10.151771
delivered,8,0.269815
created,5,0.168634
approved,2,0.067454


Of the 2,965 orders without a delivery date, 2,957 make sense: they have not been delivered yet (shipped, invoiced, processing, created, approved) or never will be (canceled, unavailable). The exception is 8 orders with order_status = "delivered". I will keep them because they contribute to revenue, and they are automatically excluded from delivery-time metrics since their delivery date is missing. One possible explanation is that the delivery was not recorded, or that the orders were lost on the way to the customer, but the data does not confirm this.


In [6]:
delivered_without_data = orders.loc[(orders["order_status"] == "delivered") & (orders["order_delivered_customer_date"].isna()), ["order_status", "order_delivered_customer_date"]]
print(delivered_without_data)
print(f"Total: {len(delivered_without_data)} orders")

print("\n")

no_delivered_with_data = orders.loc[(orders["order_status"] != "delivered") & (~ orders["order_delivered_customer_date"].isna()), ["order_status", "order_delivered_customer_date"]]
print(no_delivered_with_data)
print(f"Total: {len(no_delivered_with_data)} orders")

      order_status order_delivered_customer_date
3002     delivered                           NaT
20618    delivered                           NaT
43834    delivered                           NaT
79263    delivered                           NaT
82868    delivered                           NaT
92643    delivered                           NaT
97647    delivered                           NaT
98038    delivered                           NaT
Total: 8 orders


      order_status order_delivered_customer_date
2921      canceled           2018-03-21 22:03:51
8791      canceled           2016-10-16 14:36:59
58266     canceled           2016-10-14 15:07:11
59332     canceled           2016-10-19 18:47:43
92636     canceled           2016-11-08 10:58:34
94399     canceled           2016-11-09 14:53:50
Total: 6 orders


I will keep the 6 canceled orders that have a delivery date, but all canceled orders will be excluded from the revenue calculation. These could be orders that were returned after delivery.

In [7]:
products = dados["products"]

cols = ["product_category_name", "product_name_lenght",
        "product_description_lenght", "product_photos_qty"]

print(products.isna().sum())
print("\n")
print(products[cols].isna().all(axis=1).sum())



product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


610


The 610 products with a null product_category_name also have nulls in name length, description length and photos quantity, so they are products with no metadata. I will fill the category with "unknown". The 2 products with null weight and dimensions will stay as NaN: filling them with 0 would create false data and pull the mean down.

In [8]:
order_approved_Nat = orders[orders["order_approved_at"].isna()]
order_approved_Nat["order_status"].value_counts()


order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

Null values in order_approved_at make sense for canceled orders (141), which were canceled before approval, and for created orders (5), which are still being created. However, the 14 delivered orders should have an approval date, because an order cannot be delivered without being approved. We can also conclude that, of the 625 canceled orders (619 + 6), only 141 were canceled before being approved.

In [9]:
order_delivered_carrier_without_date = orders[orders["order_delivered_carrier_date"].isna()]
count = order_delivered_carrier_without_date["order_status"].value_counts()
print(count)



order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64


All statuses make sense with a null order_delivered_carrier_date, except for delivered (2 orders): an order that was delivered must have been handed over to the carrier. We can also see that only 75 canceled orders (625 − 550) reached the carrier before being canceled.

In [31]:
order_review = dados["order_reviews"]

print(order_review.isna().sum())




review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


Only "review_comment_title" and "review_comment_message" have null values. This is expected, because leaving a comment is optional when submitting a review. The key column, review_score, has no nulls.

In [38]:
delivered = orders[orders["order_status"] == "delivered"]
col = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]

soma = delivered[col].isna().any(axis=1).sum()

print("\n")
print(f"Percentage of orders that have a delivered order status but with null values at col: {round(soma/len(delivered) * 100, 2)}")

null_sum = delivered[col].isna().sum(axis=1)
orders.loc[null_sum[null_sum > 1].index, ["order_id", "order_delivered_carrier_date", "order_delivered_customer_date","order_approved_at"]]

order_review[order_review["order_id"] == "2d858f451373b04fb5c984a1cc2defaf"]



Percentage of orders that have a delivered order status but with null values at col: 0.02


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
51996,4e755f114e50d33b9ac6a56e0d7d3ea9,2d858f451373b04fb5c984a1cc2defaf,5,NaN,NaN,2017-06-25,2017-06-27 01:49:04
